# 비선형 차원축소 실습

**t-SNE · UMAP · Manifold Learning**

고차원 데이터의 국소 이웃 관계를 보존하며 저차원으로 시각화하는 방법. 축의 거리와 크기는 해석에 주의가 필요하다.

소재 분야에서 이해하기: 결정 구조 임베딩을 2차원 지도로 그려 군집을 확인한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [scikit-learn 매니폴드 학습 문서](https://scikit-learn.org/stable/modules/manifold.html)

## 1. 휘어진 구조를 펼치기

3차원에서 말려 있는 데이터를 2차원으로 펼쳐봅니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

t = rng.uniform(1.5 * np.pi, 4.5 * np.pi, 800)
height = rng.uniform(0, 12, 800)
roll = np.column_stack([t * np.cos(t), height, t * np.sin(t)])
figure = plt.figure(figsize=(5, 4))
axis = figure.add_subplot(111, projection='3d')
axis.scatter(roll[:, 0], roll[:, 1], roll[:, 2], c=t, cmap='viridis', s=6)
axis.set_xlabel('x'); axis.set_ylabel('y'); axis.set_zlabel('z'); plt.show()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, Isomap

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for axis, (name, model) in zip(axes, [('PCA', PCA(2)),
                                      ('Isomap', Isomap(n_neighbors=12, n_components=2)),
                                      ('t-SNE', TSNE(2, perplexity=30, random_state=0))]):
    low = model.fit_transform(roll)
    axis.scatter(low[:, 0], low[:, 1], c=t, cmap='viridis', s=6)
    axis.set_title(name)
plt.tight_layout(); plt.show()
print('색이 매끄럽게 이어지면 원래의 이웃 관계가 보존된 것입니다.')

## 2. 축과 거리를 해석하면 안 되는 이유

In [ ]:
for perplexity in (5, 30, 80):
    low = TSNE(2, perplexity=perplexity, random_state=0).fit_transform(roll)
    spread = low.max(0) - low.min(0)
    print('perplexity %2d -> 그림의 가로세로 범위 %s (설정마다 크기와 모양이 달라집니다)'
          % (perplexity, np.round(spread, 1)))
print('\nt-SNE/UMAP 그림에서 군집 간 거리와 군집 크기는 원 데이터의 거리와 비례하지 않습니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#manifold-learning)을 여세요.